## Задание 2. Алгоритм обратного распространения ошибки

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

def Loss(y_pred, y):
    y_pred = y_pred.reshape(-1, 1)
    y = np.array(y).reshape(-1, 1)
    return 0.5 * np.mean((y_pred - y) ** 2)

def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(float)

class NeuronReLU:
    def __init__(self, w=None, b=0):
        self.w = w
        self.b = b

    def activate(self, x):
        return relu(x)

    def forward_pass(self, X):
        return self.activate(X @ self.w + self.b)

    def backward_pass(self, X, y, y_pred, learning_rate=0.005):
        n = len(y)
        y = np.array(y).reshape(-1, 1)
        delta = (1 / n) * (y_pred - y) * relu_derivative(X @ self.w + self.b)
        self.w -= learning_rate * (X.T @ delta)
        self.b -= learning_rate * np.sum(delta)

    def fit(self, X, y, num_epochs=300):
        Loss_values = []
        for i in range(num_epochs):
            y_pred = self.forward_pass(X)
            Loss_values.append(Loss(y_pred, y))
            self.backward_pass(X, y, y_pred)
        return Loss_values

## Задание 3. Нейрон с различными функциями активации

In [ ]:
def leaky_relu(x, alpha=0.01):
    return np.where(x > 0, x, alpha * x)

def leaky_relu_derivative(x, alpha=0.01):
    return np.where(x > 0, 1.0, alpha)

def elu(x, alpha=0.01):
    return np.where(x > 0, x, alpha * (np.exp(x) - 1))

def elu_derivative(x, alpha=0.01):
    return np.where(x > 0, 1.0, elu(x, alpha) + alpha)

class ActivationNeuron:
    def __init__(self, activation, activation_derivative, w=None, b=0):
        self.activation = activation
        self.activation_derivative = activation_derivative
        self.w = w
        self.b = b

    def forward_pass(self, X):
        return self.activation(X @ self.w + self.b)

    def backward_pass(self, X, y, y_pred, learning_rate=0.005):
        y = np.array(y).reshape(-1, 1)
        z = X @ self.w + self.b
        delta = (2 / len(y)) * (y_pred - y) * self.activation_derivative(z)
        self.w -= learning_rate * (X.T @ delta)
        self.b -= learning_rate * np.sum(delta)

    def fit(self, X, y, num_epochs=300, learning_rate=0.005):
        if self.w is None: self.w = np.random.rand(X.shape[1], 1)
        losses = []
        for _ in range(num_epochs):
            y_pred = self.forward_pass(X)
            losses.append(Loss(y_pred, y))
            self.backward_pass(X, y, y_pred, learning_rate=learning_rate)
        return losses